# Daily Challenge: Building Your First Neural Network on MNIST
## Week 5 — Day 5 | DI GenAI & Machine Learning Bootcamp 2026

**Objective:** Build, train and evaluate a fully connected neural network to classify handwritten digits (0–9) from the MNIST dataset.

**Steps:**
1. Load & preprocess MNIST
2. Build a fully connected neural network (2 hidden layers)
3. Train for 10 epochs
4. Evaluate: accuracy + confusion matrix + hardest digits

## Step 1 — Imports & Setup

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow {tf.__version__} ✓")
print(f"NumPy {np.__version__} ✓")

## Step 2 — Load & Preprocess the MNIST Dataset

In [ ]:
# Load dataset (already split into train/test)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Training set:  {X_train.shape[0]} images — shape {X_train.shape[1:]}")
print(f"Test set:      {X_test.shape[0]} images  — shape {X_test.shape[1:]}")
print(f"Classes:       {np.unique(y_train)} (digits 0–9)")
print(f"Pixel range:   [{X_train.min()}, {X_train.max()}]")

# Normalize pixel values to [0, 1]
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0
print(f"\nAfter normalization: [{X_train_norm.min():.1f}, {X_train_norm.max():.1f}] ✓")

# One-hot encode labels  (e.g. 5 → [0,0,0,0,0,1,0,0,0,0])
y_train_ohe = to_categorical(y_train, num_classes=10)
y_test_ohe  = to_categorical(y_test,  num_classes=10)
print(f"Label example:  {y_train[0]}  →  {y_train_ohe[0]}")

In [ ]:
# Display 20 sample images with their labels
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i], cmap='gray_r')
    ax.set_title(f"Label: {y_train[i]}", fontsize=9, fontweight='bold')
    ax.axis('off')
plt.suptitle('MNIST — Sample Training Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dc_plot1_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Step 3 — Build the Fully Connected Neural Network

In [ ]:
model = models.Sequential([
    # Flatten 28×28 image into 784-pixel vector
    layers.Flatten(input_shape=(28, 28)),

    # Hidden layer 1: 256 neurons + ReLU + Dropout to reduce overfitting
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),

    # Hidden layer 2: 128 neurons + ReLU
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),

    # Output layer: 10 neurons (one per digit) + Softmax
    layers.Dense(10, activation='softmax')
], name='MNIST_Digit_Classifier')

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 4 — Train the Neural Network (10 epochs)

In [ ]:
history = model.fit(
    X_train_norm, y_train_ohe,
    epochs=10,
    batch_size=32,
    validation_split=0.1,   # 10% of training data used for validation
    verbose=1
)

In [ ]:
# Plot training curves — Loss & Accuracy over epochs
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train', color='#4e91d4', lw=2.5)
axes[0].plot(history.history['val_accuracy'], label='Validation', color='#e05c5c', lw=2.5, linestyle='--')
axes[0].set_title('Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.9, 1.0])

# Loss
axes[1].plot(history.history['loss'],     label='Train', color='#4e91d4', lw=2.5)
axes[1].plot(history.history['val_loss'], label='Validation', color='#e05c5c', lw=2.5, linestyle='--')
axes[1].set_title('Loss over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History — MNIST Neural Network', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dc_plot2_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Step 5 — Evaluate Model Performance

In [ ]:
# Test accuracy
test_loss, test_acc = model.evaluate(X_test_norm, y_test_ohe, verbose=0)
print(f"{'='*40}")
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"{'='*40}")

# Predictions
y_pred_proba = model.predict(X_test_norm, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = y_test

# Per-digit accuracy
print("\nPer-digit accuracy:")
print(f"{'Digit':<8} {'Correct':<10} {'Total':<8} {'Accuracy'}")
print("-" * 38)
for digit in range(10):
    mask    = y_true == digit
    correct = np.sum(y_pred[mask] == digit)
    total   = np.sum(mask)
    acc     = correct / total * 100
    bar     = "█" * int(acc / 5)
    print(f"  {digit:<6} {correct:<10} {total:<8} {acc:.1f}%  {bar}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10),
            linewidths=0.5, annot_kws={'size': 10})
plt.title('Confusion Matrix — MNIST Neural Network', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Digit', fontsize=12)
plt.ylabel('True Digit', fontsize=12)
plt.tight_layout()
plt.savefig('dc_plot3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify most confused pairs
print("\nTop 5 most confused digit pairs:")
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
confused = []
for i in range(10):
    for j in range(10):
        if i != j and cm_no_diag[i, j] > 0:
            confused.append((cm_no_diag[i, j], i, j))
confused.sort(reverse=True)
for count, true, pred in confused[:5]:
    print(f"  True={true} predicted as {pred}: {count} times")
print("Plot saved ✓")

In [ ]:
# Visualize misclassified examples
wrong_idx = np.where(y_pred != y_true)[0]
print(f"Total misclassified: {len(wrong_idx)} / {len(y_true)}")

fig, axes = plt.subplots(3, 6, figsize=(15, 8))
axes = axes.flatten()
for i, idx in enumerate(wrong_idx[:18]):
    axes[i].imshow(X_test[idx], cmap='gray_r')
    axes[i].set_title(
        f"True: {y_true[idx]}\nPred: {y_pred[idx]}\n({y_pred_proba[idx][y_pred[idx]]*100:.1f}%)",
        fontsize=8, color='#e05c5c', fontweight='bold'
    )
    axes[i].axis('off')

plt.suptitle('Misclassified Digits — Where the Model Fails', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dc_plot4_misclassified.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Bonus — Hyperparameter Tuning Experiment

Compare 3 configurations to see which gives the best accuracy.

In [ ]:
configs = [
    {"name": "Baseline",       "units": [64, 32],   "dropout": 0.0, "lr": 0.001},
    {"name": "Deeper",         "units": [256, 128],  "dropout": 0.2, "lr": 0.001},
    {"name": "High Dropout",   "units": [256, 128],  "dropout": 0.4, "lr": 0.0005},
]

results = []
for cfg in configs:
    tf.random.set_seed(42)
    m = models.Sequential(name=cfg['name'].replace(' ', '_'))
    m.add(layers.Flatten(input_shape=(28, 28)))
    for units in cfg['units']:
        m.add(layers.Dense(units, activation='relu'))
        if cfg['dropout'] > 0:
            m.add(layers.Dropout(cfg['dropout']))
    m.add(layers.Dense(10, activation='softmax'))
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']),
              loss='categorical_crossentropy', metrics=['accuracy'])
    hist = m.fit(X_train_norm, y_train_ohe, epochs=5, batch_size=64,
                 validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_test_norm, y_test_ohe, verbose=0)
    results.append({'Config': cfg['name'], 'Val Acc (ep5)': f"{hist.history['val_accuracy'][-1]*100:.2f}%",
                    'Test Acc': f"{acc*100:.2f}%"})
    print(f"  {cfg['name']:<20} → Test Accuracy: {acc*100:.2f}%")

import pandas as pd
print("\n")
print(pd.DataFrame(results).to_string(index=False))